In [10]:
!pip install transformers torch sentencepiece rouge-score sumy

In [11]:
import torch
from transformers import PegasusForConditionalGeneration, PegasusTokenizer
from rouge_score import rouge_scorer
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer
import nltk

nltk.download('punkt', quiet=True)

print("Loading google/pegasus-xsum model...")
model_name = "google/pegasus-xsum"
tokenizer = PegasusTokenizer.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = PegasusForConditionalGeneration.from_pretrained(model_name).to(device)

def get_summary(text, beams=4, penalty=1.0):
    """Encodes input text and generates abstractive summary using Pegasus."""
    inputs = tokenizer(text, truncation=True, padding="longest", return_tensors="pt").to(device)

    outputs = model.generate(
        inputs["input_ids"],
        num_beams=beams,
        length_penalty=penalty,
        max_length=64,
        min_length=10
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Model successfully loaded.")

Loading google/pegasus-xsum model...


Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-xsum
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model successfully loaded.


In [12]:
articles = {
    "mars_rover": (
        "The European Space Agency just announced that its Horizon rover safely touched down in the Jezero Crater "
        "on Mars after a brutal seven-month flight. The team back in Germany completely lost their minds cheering "
        "when the first high-res telemetry images pinged back. Everything is running perfectly right now. Horizon is "
        "going to spend the next two Earth years drilling for soil samples to see if there was ever ancient microbial "
        "life in what used to be a massive lake system. Interestingly, this is the first time a rover is using fully "
        "autonomous AI to drive itself around tricky rocks without having to wait hours for commands from Earth. "
        "NASA is also helping out by letting them use their deep-space tracking network for communication."
    ),
    "climate_summit": (
        "World leaders at the UN Climate Summit just signed a massive, historic deal in Geneva after pulling three "
        "straight all-nighters debating details. The headline is that they are legally binding everyone to phase out "
        "coal power completely by 2040. They're also setting up a trillion-dollar 'Global Green Resilience Fund' "
        "where rich G7 nations and corporate carbon offsets will help developing countries handle climate disasters. "
        "Activists are happy but pointing out that enforcement is still pretty vague. Some industrial giants complained "
        "about their economies taking a hit but eventually gave in after getting green transition subsidies. Island "
        "nations are still terrified that 2040 is simply too slow to stop rising ocean levels."
    ),
    "crispr_blindness": (
        "Researchers over at the Tokyo Institute of Technology just published a crazy study showing a 90% success rate "
        "using CRISPR gene-editing to reverse blindness in adults. They tracked 50 patients who were legally blind "
        "due to a genetic mutation that wrecks retinal cells. After just one tiny injection of the CRISPR package into "
        "the eye, most patients started seeing shapes, colors, and actual text within three months. Dr. Kenji Sato, "
        "who led the trial, noted they didn't see any weird side effects or accidental mutations. Now, bioethicists "
        "are sounding alarms about how expensive this is going to be when it hits the market, but if regulators fast-track "
        "it, the therapy could be legally available in three years."
    )
}
print("Dataset initialized.")

Dataset initialized.


In [13]:
print("=== Part A: Generating Basic Summaries ===")
ai_summaries = {}

for key, text in articles.items():
    summary = get_summary(text)
    ai_summaries[key] = summary

    orig_len = len(text.split())
    sum_len = len(summary.split())

    print(f"\n[{key.upper()}] (Words: {orig_len} -> {sum_len})")
    print(f"Generated Summary: {summary}")

=== Part A: Generating Basic Summaries ===

[MARS_ROVER] (Words: 123 -> 7)
Generated Summary: It's been a long, long time coming.

[CLIMATE_SUMMIT] (Words: 111 -> 9)
Generated Summary: It's the deal that's been years in the making.

[CRISPR_BLINDNESS] (Words: 117 -> 10)
Generated Summary: The world's first gene-edited eye could be on its way.


In [14]:
print("=== Part B: Parameter Exploration ===")
mars_text = articles["mars_rover"]

print("\n--- 1. Testing Beam Search (num_beams) ---")
for b in [1, 4, 8]:
    res = get_summary(mars_text, beams=b, penalty=1.0)
    print(f"Beams {b} | Words: {len(res.split())} | {res}")

print("\n--- 2. Testing Length Penalty ---")
for p in [0.8, 1.0, 2.0]:
    res = get_summary(mars_text, beams=4, penalty=p)
    print(f"Penalty {p} | Words: {len(res.split())} | {res}")

=== Part B: Parameter Exploration ===

--- 1. Testing Beam Search (num_beams) ---
Beams 1 | Words: 7 | It's a big day for Mars exploration.
Beams 4 | Words: 7 | It's been a long, long time coming.
Beams 8 | Words: 10 | It's all systems go for Europe's latest mission to Mars.

--- 2. Testing Length Penalty ---
Penalty 0.8 | Words: 6 | It's been a long time coming.
Penalty 1.0 | Words: 7 | It's been a long, long time coming.
Penalty 2.0 | Words: 11 | It's been a long, long time coming, but it's finally over.


In [15]:
# Added 'punkt_tab' here to fix the LookupError
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("Loading google/pegasus-xsum model...")
model_name = "google/pegasus-xsum"
tokenizer = PegasusTokenizer.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = PegasusForConditionalGeneration.from_pretrained(model_name).to(device)

def get_summary(text, beams=4, penalty=1.0):
    """Encodes input text and generates abstractive summary using Pegasus."""
    inputs = tokenizer(text, truncation=True, padding="longest", return_tensors="pt").to(device)

    outputs = model.generate(
        inputs["input_ids"],
        num_beams=beams,
        length_penalty=penalty,
        max_length=64,
        min_length=10
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Model successfully loaded.")

Loading google/pegasus-xsum model...


Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-xsum
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model successfully loaded.
